# Pauli Collision Entropy and Smooth Min-Entropy

Adapted from the supplied Werner notebook for
$$\rho_{XB}=\tfrac12\sum_{x=0}^1 |x\rangle\langle x|\otimes|\phi^x\rangle\langle\phi^x|,
\qquad |\phi^x\rangle=\sqrt p|0\rangle+(-1)^x\sqrt{1-p}|1\rangle.$$
The default is **p = 0.05**, as in
[Tomamichel and Hayashi, arXiv:1208.1478](https://arxiv.org/html/1208.1478#A1).

Open the notebook from the repository root and run all cells to load the bundled checkpoints, summarize their coverage, and
plot the measured smooth collision entropy alongside the smooth min-entropy.
The default view uses the saved MOSEK series at p = 0.05.

**Smoothing conventions:** epsilon is the linear penalty parameter in the
supplied measured-collision program. To retain its min-entropy comparison,
we use the purified-distance radius delta = sqrt(epsilon). The radius
and its square are stored explicitly in the min-entropy results;
entropy curves are not themselves secret-key lengths. One global smoothing
budget is used for all n copies.

Copy permutation symmetry reduces the collision matrix to size n+1.
The conditional pure states reduce smooth min-entropy to a scalar search.

Set `RUN_SWEEPS = True` in **Parameters and Solver Settings** to extend the
sweeps. Edit the copy counts and smoothing values there before running.
Analysis and new checkpoints are written to `results/` by default.
For batch jobs, use the PBS launcher or command-line runner in this repository. A local checkpoint takes precedence over the corresponding
original CSV when present.

In [ ]:
# 02. Results Folders
import os
from pathlib import Path

from IPython.display import display

%matplotlib inline
PROJECT_DIR = Path.cwd()
SAVED_RESULTS_DIR = PROJECT_DIR / 'pauli_entropy_results'
OUTPUT_DIR = Path(os.environ.get('PAULI_OUTPUT_DIR', PROJECT_DIR / 'results')).resolve()

print('Saved checkpoints:', SAVED_RESULTS_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
# @title 03. Imports and Available Solvers
import hashlib
import itertools
import json
import math
import os
import time
import warnings
from pathlib import Path
from functools import lru_cache
from math import comb, lgamma
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
from scipy.special import gammaln, logsumexp
from scipy.stats import binom

# Required packages: numpy pandas cvxpy clarabel scipy matplotlib.
# MOSEK is only needed when running an optimization with that solver.
print('Available solvers:', cp.installed_solvers())
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False})

In [ ]:
# @title 04. Parameters and Solver Settings
P_VALUES = [0.05]
COLLISION_EPSILON_VALUES = [0.1, 0.01, 0.001, 0.0001, 0.00001, 0.000001, 0.0]
MIN_ENTROPY_EPSILON_VALUES = list(COLLISION_EPSILON_VALUES)
COLLISION_N_VALUES = list(range(1, 226))
MIN_ENTROPY_N_VALUES = list(range(1, 226))
COLLISION_SOLVER = 'MOSEK'  # Saved series; CLARABEL or SCS can be used for new sweeps.
COLLISION_TOL = 1e-5
MAX_COLLISION_COPIES = 1000000  # Numerical guard for the n+1 matrix; increase after checking residuals
MAX_BASELINE_COPIES = 2  # Full exponential matrices are used only for validation
MAX_SCALAR_MIN_COPIES = 1000000  # Guard against loss of objective precision
MOSEK_THREADS = int(os.environ.get('PAULI_MOSEK_THREADS', '4'))
if MOSEK_THREADS < 1:
    raise ValueError('PAULI_MOSEK_THREADS must be positive.')
COLLISION_FORMULATION = 'normalized-gauge-v1'
SOLVER_VERBOSE = False
RUN_SWEEPS = False  # Set True to run the selected grid.
REPAIR_COLLISION_CHECKPOINTS = False  # Retry failures and flagged collision rows in the saved series.
RESUME = True
CONTINUE_ON_ERROR = True
RUN_VALIDATION = False  # Enable after changing a formulation.
VALIDATION_N_VALUES = [1, 2]
VALIDATION_P_VALUES = [0.0, 0.05, 0.25, 0.5, 0.95, 1.0]
VALIDATION_EPSILON_VALUES = [0.0, 0.01, 0.1]
VALIDATION_RTOL = 3e-4
VALIDATION_ATOL = 2e-7
MODEL_VERSION = 'pauli-pure-cq-v3-single-radius'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Run new sweeps:', RUN_SWEEPS, '| Retry saved collision rows:', REPAIR_COLLISION_CHECKPOINTS,
      '| Run baseline validation:', RUN_VALIDATION)

In [ ]:
# @title 05. Shared Validation, Checkpoint, and Plot Helpers
def checked_values(values, name, kind='probability'):
    values = list(values)
    if not values:
        raise ValueError(f'{name} must contain at least one value.')
    for value in values:
        if kind == 'copies':
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
                raise ValueError(f'{name} must contain positive integers.')
        elif not np.isfinite(value) or not 0 <= value <= 1 or (kind == 'epsilon' and value == 1):
            raise ValueError(f'Invalid value {value!r} in {name}.')
    return list(dict.fromkeys(values))


def require_solver(solver):
    if solver.upper() not in cp.installed_solvers():
        raise RuntimeError(f'Solver {solver} is unavailable: {cp.installed_solvers()}')


def solver_options(solver, tol):
    if not np.isfinite(tol) or tol <= 0:
        raise ValueError('tol must be positive and finite.')
    solver = solver.upper()
    if solver == 'CLARABEL':
        return dict(tol_gap_abs=tol, tol_gap_rel=tol, tol_feas=tol, max_iter=500)
    if solver == 'SCS':
        return dict(eps=tol, max_iters=200000)
    if solver == 'MOSEK':
        return dict(mosek_params={'MSK_DPAR_INTPNT_CO_TOL_REL_GAP': tol})
    return {}


def collision_residuals_ok(rows):
    limit = 100 * rows['tol']
    psd = rows.get('psd_violation', np.nan)
    norm = rows.get('norm_violation', np.nan)
    return np.isfinite(psd) & np.isfinite(norm) & (psd <= limit) & (norm <= limit)


def checkpoint_key(calculation, case):
    payload = dict(model_version=MODEL_VERSION, calculation=calculation, **case)
    return hashlib.sha256(json.dumps(payload, sort_keys=True, allow_nan=False).encode()).hexdigest()


def write_checkpoint(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + '.tmp')
    pd.DataFrame(list(records.values())).to_csv(temporary_path, index=False, float_format='%.17g')
    os.replace(temporary_path, path)


def checkpoint_source(calculation):
    name = f'{calculation}_results.csv'
    local_path = OUTPUT_DIR / name
    return local_path if local_path.is_file() else SAVED_RESULTS_DIR / name


def read_checkpoint(path):
    frame = pd.read_csv(path, float_precision='round_trip')
    if not {'case_key', 'run_status', 'model_version'}.issubset(frame.columns):
        raise ValueError(f'Unrecognized checkpoint schema: {path}')
    return frame.loc[frame['model_version'].eq(MODEL_VERSION)].copy()


def run_sweep(calculation, grid, solve_case):
    cases = list(grid)
    if not cases:
        raise ValueError(f'{calculation}: the parameter grid is empty.')
    path = Path(OUTPUT_DIR) / f'{calculation}_results.csv'
    records = {}
    source = checkpoint_source(calculation)
    if RESUME and source.is_file():
        previous = read_checkpoint(source)
        if 'error' in previous:
            previous['error'] = previous['error'].fillna('')
        records = {row['case_key']: row for row in previous.to_dict('records')}
    current_rows = []
    for index, case in enumerate(cases, 1):
        key = checkpoint_key(calculation, case)
        row = records.get(key, {})
        reusable = row.get('run_status') == 'ok'
        if reusable and calculation == 'collision':
            reusable = (row.get('collision_formulation') == COLLISION_FORMULATION
                        and row.get('solver_status') in ('optimal', 'analytic_endpoint')
                        and collision_residuals_ok(row))
        if reusable:
            row.update(case)  # Preserve exact requested parameters when reusing CSV rows.
            print(f'[{index}/{len(cases)}] {calculation}: reused {case}')
        else:
            print(f'[{index}/{len(cases)}] {calculation}: solving {case}', flush=True)
            started = time.perf_counter()
            row = dict(case_key=key, model_version=MODEL_VERSION, **case)
            try:
                row.update(solve_case(case))
                row.update(run_status='ok', error='', wall_seconds=time.perf_counter() - started)
            except Exception as error:
                row.update(run_status='error', error=f'{type(error).__name__}: {error}',
                           wall_seconds=time.perf_counter() - started)
                records[key] = row
                write_checkpoint(records, path)
                print(f"  Failed: {row['error']}", flush=True)
                if not CONTINUE_ON_ERROR:
                    raise
            records[key] = row
            write_checkpoint(records, path)
        current_rows.append(row)
    write_checkpoint(records, path)
    frame = pd.DataFrame(current_rows)
    print(f"{calculation}: {(frame['run_status'] == 'ok').sum()}/{len(frame)} successful; {path}")
    return frame


def successful_results(frame, name):
    if frame is None or frame.empty:
        print(f'{name}: no results available.')
        return pd.DataFrame()
    good = frame.loc[frame['run_status'] == 'ok'].copy()
    if good.empty:
        print(f'{name}: no successful results. Inspect the error column in its CSV.')
    return good


def validate_parameters(p, n, epsilon=None, matrix=False, collision=False):
    if not np.isfinite(p) or not 0 <= p <= 1:
        raise ValueError('Require 0 <= p <= 1.')
    if isinstance(n, (bool, np.bool_)) or not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError('n must be a positive integer.')
    if epsilon is not None and (not np.isfinite(epsilon) or not 0 <= epsilon < 1):
        raise ValueError('Require 0 <= smoothing parameter < 1.')
    limit = MAX_BASELINE_COPIES if matrix else (MAX_COLLISION_COPIES if collision else MAX_SCALAR_MIN_COPIES)
    if n > limit:
        raise ValueError(f'n={n} exceeds allocation/numerical guard {limit}.')

def quantum_grid(epsilon_values, n_values):
    # Keep epsilon values together so one compiled collision problem can be reused.
    for p, n, epsilon in itertools.product(
        checked_values(P_VALUES, 'P_VALUES'),
        checked_values(n_values, 'copy counts', 'copies'),
        checked_values(epsilon_values, 'epsilon values', 'epsilon')):
        yield dict(p=float(p), epsilon=float(epsilon), n=int(n),
                   solver=COLLISION_SOLVER, tol=COLLISION_TOL)

def plot_curve(curve):
    # Break the line at each gap without allocating rows for every missing n.
    curve = curve.set_index('n').sort_index()
    gap_end = curve.index[curve.index.to_series().diff().gt(1)]
    return curve.reindex(curve.index.union(gap_end - 1)).reset_index()


def save_figure(fig, name):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f'{name}.png', dpi=180, bbox_inches='tight')
    plt.show()

# @title 06. State Construction and Analytic Reference Values

Write $Z=\operatorname{diag}(1,-1)$ and
$R=|\phi^0\rangle\langle\phi^0|=\begin{pmatrix}p&\sqrt{p(1-p)}\\\sqrt{p(1-p)}&1-p\end{pmatrix}$.
Then the one-copy CQ blocks are $\tau_0=R/2$ and $\tau_1=ZRZ/2$.
For n copies, $\tau_x=2^{-n}Z_xR^{\otimes n}Z_x$ and
$\rho_{B^n}=\operatorname{diag}(p,1-p)^{\otimes n}$.

Directly from these matrices,
$$H(X|B)=1-h_2(p),\quad
V(X|B)=p(1-p)\left[\log_2\frac{1-p}{p}\right]^2,$$
$$H_{\min}(X^n|B^n)=-n\log_2\frac{1+2\sqrt{p(1-p)}}{2}.$$
At p = 0 and 1, B has no information about X; at p = 1/2, the two
conditional states are orthogonal and B determines X perfectly.


In [ ]:
# @title 07. State Constructors and Reference Calculations
def pauli_conditional_states(p):
    validate_parameters(p, 1)
    v = np.array([np.sqrt(p), np.sqrt(1-p)])
    z = np.array([1., -1.])
    return np.outer(v, v), np.outer(v*z, v*z)

def generate_pauli_cq_blocks(p, n=1):
    validate_parameters(p, n, matrix=True)
    blocks = [r/2 for r in pauli_conditional_states(p)]
    single = list(blocks)
    for _ in range(n-1):
        blocks = [np.kron(b, s) for b in blocks for s in single]
    return blocks

def seed_vector(p, n):
    validate_parameters(p, n, matrix=True)
    v = np.ones(1)
    for _ in range(n):
        v = np.kron(v, [np.sqrt(p), np.sqrt(1-p)])
    return v

def h2(p):
    validate_parameters(p, 1)
    return -sum(t*math.log2(t) for t in (p, 1-p) if t > 0)

def pauli_conditional_entropy(p):
    return 1-h2(p)

def pauli_information_variance(p):
    validate_parameters(p, 1)
    return 0.0 if p in (0, 1) else p*(1-p)*math.log2((1-p)/p)**2

def unsmoothed_min_entropy(p, n=1):
    validate_parameters(p, n)
    return -n*math.log2((1+2*math.sqrt(p*(1-p)))/2)

reference_results = pd.DataFrame([
    dict(p=p, H_vN_bits=pauli_conditional_entropy(p), V_bits_squared=pauli_information_variance(p),
         Hmin_unsmoothed_bits=unsmoothed_min_entropy(p))
    for p in P_VALUES])
print(reference_results.to_string(index=False))

@lru_cache(maxsize=128)
def hamming_vectors(p, n):
    """Immutable sector weights, shared across epsilon values and both entropies."""
    k = np.arange(n+1)
    log_m = gammaln(n+1)-gammaln(k+1)-gammaln(n-k+1)
    u = np.exp(0.5*binom.logpmf(k, n, p))
    u /= np.linalg.norm(u)
    c = np.exp(0.5*(log_m-n*math.log(2)))
    inv_sqrt_m = np.exp(-0.5*log_m)
    for array in (u, c, inv_sqrt_m):
        array.setflags(write=False)
    return u, c, inv_sqrt_m


# @title 08. Measured Collision Entropy: Exact Symmetry Reductions

This preserves **the supplied notebook's B8 optimization**, whose source is not
identified in that notebook. We compare this measured-collision quantity with smooth min-entropy.

The Z-flip symmetry permits $B_x=Z_xBZ_x$ and $C_x=Z_xCZ_x$.
The Schur constraint is $C\succeq B^2$, and the group sum removes off-diagonal
entries: $\sum_xZ_xCZ_x=2^n\operatorname{Diag}(\operatorname{diag}C)$.
With $d=2^n$, $A=dB$, $T=dt$, $K=dk$, the first reduced program is
$$F^*=\max\{2\langle v|A|v\rangle-2\epsilon T-K:
0\preceq A\preceq TI,\ \|A_{:,j}\|_2^2\le K\ \forall j\}.$$

**Copy permutations give a second exact reduction.** Averaging A over permutations
preserves the objective and feasibility. The average commutes with the projector
P onto the symmetric subspace, and v is in that subspace. Replacing A by PAP leaves
the objective unchanged. Since A commutes with P, $(PAP)^2\preceq A^2$, so none of
the squared column norms increases; both PSD bounds are also preserved.

In the normalized symmetric basis, let $m_k=\binom nk$ and
$u_k=\sqrt{m_kp^k(1-p)^{n-k}}$, with k counting the component of amplitude sqrt(p).
The matrix S has size **n+1**, and the equivalent program is
$$F^*=\max\{2u^TSu-2\epsilon T-K:
0\preceq S\preceq TI,\ \|S_{:,k}/\sqrt{m_k}\|_2^2\le K\ \forall k\}.$$
Recover $q_\epsilon=F^*/2^n$ and $H_2^{\epsilon,\mathbb M}=n-\log_2F^*$.
The matrix still describes a collective optimization across all copies.


### Numerically scaled solve

For any nonzero solution, divide the matrix and its spectral bound by
$\sqrt{2^n K}$. Equivalently, set $K=1$ before dividing by $\sqrt{2^n}$.
With $c_k=\sqrt{m_k/2^n}$, solve
$$L^*=\max\{u^TAu-\epsilon\lambda_{\max}(A):
A\succeq0,\ \|A_{:,k}\|_2\le c_k\ \forall k\}.$$
Optimizing the overall scale in the original quadratic objective gives
$q_\epsilon=(L^*)^2$ and $H_2^{\epsilon,\mathbb M}=-2\log_2L^*$.
The spectral bound is omitted when epsilon is zero.

The solver scales the matrix entries using their column caps and balances
the objective and the two positive-semidefinite constraints separately.
Endpoint states $p=0,1/2,1$ have analytic solutions.

**Bounded sector omission.** Every feasible entry obeys
$|A_{ij}|\le\min(c_i,c_j)$. Restricting a feasible matrix to retained
sectors preserves feasibility and can only reduce its spectral penalty.
If $D$ is the set of omitted sectors, the reduction in optimal $L$ is
therefore bounded above by
$$E=2\sum_{i\in D}u_i\sum_j u_j\min(c_i,c_j).$$
The code chooses $D$ using a feasible diagonal lower bound. It reports the
resulting entropy bound $2\log_2(1+E/L)$ separately from solver accuracy.
For recovery, it corrects any small PSD violation in the scaled matrix,
enforces the column caps, and evaluates the actual spectral penalty.

In [ ]:
# @title 09. Numerically Scaled Collision Entropy Solver
def collision_sectors(p, n, epsilon, relative_tail_budget):
    u, c, _ = hamming_vectors(p, n)
    if np.any(c <= 0):
        raise ValueError('A collision sector cap underflowed. Reduce the copy count.')

    # In increasing cap order, sum min(c[i], c[j]) * weights[j] without a dense matrix.
    order = np.argsort(c)
    u, c = u[order], c[order]
    tail_u = np.r_[np.cumsum(u[:0:-1])[::-1], 0.]
    weights = u**2
    tail_weights = np.r_[np.cumsum(weights[:0:-1])[::-1], 0.]
    diagonal_values = np.cumsum(c*weights) + c*(tail_weights - epsilon)
    diagonal_lower = max(c[0]*(1-epsilon), float(diagonal_values.max()))
    sector_bounds = 2*u*(np.cumsum(c*u) + c*tail_u)
    keep = sector_bounds > relative_tail_budget*diagonal_lower/len(c)
    return u[keep], c[keep], float(sector_bounds[~keep].sum())


def solve_pauli_collision(p=0.05, epsilon=1e-3, n=1, solver='CLARABEL', tol=1e-8, verbose=False):
    validate_parameters(p, n, epsilon, collision=True)
    solver = solver.upper()
    require_solver(solver)
    solver_options(solver, tol)  # Validate the requested tolerance before tightening it.
    started = time.perf_counter()
    p = float(min(p, 1-p))  # Exchange basis labels; the entropy is unchanged.
    if p in (0., 0.5):
        H = (n if p == 0. else 0.)-2*math.log1p(-epsilon)/math.log(2)
        return dict(q=2.**(-H), H_bits=H, status='analytic_endpoint',
                    solve_time=time.perf_counter()-started, compilation_seconds=0.,
                    solver_seconds=0., num_iters=0, matrix_dimension=n+1, sector_count=0,
                    psd_violation=0., norm_violation=0., tail_bound_bits=0., recovery_gap_bits=0.)

    solve_tol = min(tol, 1e-9)
    options = solver_options(solver, solve_tol)
    if solver == 'MOSEK':
        options['mosek_params'].update({
            'MSK_DPAR_INTPNT_CO_TOL_PFEAS': solve_tol,
            'MSK_DPAR_INTPNT_CO_TOL_DFEAS': solve_tol,
            'MSK_IPAR_NUM_THREADS': MOSEK_THREADS,
        })

    u, c, tail_error = collision_sectors(p, n, epsilon, min(1e-7, tol/100))
    size = len(c)
    entry_caps = np.minimum.outer(c, c)
    spectral_scale = c[-1]
    if epsilon > 0:
        # Use a scale near where the linear upper bound's slope crosses epsilon.
        crossing = np.searchsorted(np.cumsum(u), u.sum()-math.sqrt(epsilon))
        spectral_scale = min(c[min(crossing, size-1)], float(u @ entry_caps @ u)/epsilon)
        entry_caps = np.minimum(entry_caps, spectral_scale)

    diagonal_scale = np.minimum(c, spectral_scale)
    psd_coefficients = entry_caps / np.sqrt(np.outer(diagonal_scale, diagonal_scale))
    objective_weights = np.outer(u, u)*entry_caps
    objective_scale = float(objective_weights.sum())
    X = cp.Variable((size, size), symmetric=True)
    constraints = [cp.multiply(psd_coefficients, X) >> 0,
                   cp.norm(cp.multiply(entry_caps/c, X), axis=0) <= 1]
    objective = cp.sum(cp.multiply(objective_weights/objective_scale, X))
    if epsilon > 0:
        T = cp.Variable(nonneg=True)
        constraints.append(T*np.eye(size)-cp.multiply(entry_caps/spectral_scale, X) >> 0)
        objective -= (epsilon*spectral_scale/objective_scale)*T
    problem = cp.Problem(cp.Maximize(objective), constraints)
    problem.solve(solver=solver, verbose=verbose, canon_backend=cp.SCIPY_CANON_BACKEND, **options)
    if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        raise RuntimeError(f'Collision solver: {problem.status}')
    if problem.status == cp.OPTIMAL_INACCURATE:
        warnings.warn('Collision solver returned optimal_inaccurate; inspect residuals.')

    x = (X.value+X.value.T)/2
    psd_violation = max(0., -float(np.linalg.eigvalsh(psd_coefficients*x)[0]))
    a = entry_caps*x
    norm_violation = max(0., float(np.linalg.norm(a/c, axis=0).max())-1.)
    spectral_violation = 0.
    if epsilon > 0:
        spectral_violation = max(0., float(np.linalg.eigvalsh(a/spectral_scale)[-1])-float(T.value))
    # Repair the scaled PSD constraint, then enforce all column caps by a uniform contraction.
    a[np.diag_indices(size)] += diagonal_scale*psd_violation
    a /= max(1., float(np.linalg.norm(a/c, axis=0).max()))
    value = float(u @ a @ u)
    if epsilon > 0:
        value -= epsilon*float(np.linalg.eigvalsh(a)[-1])
    optimizer_value = float(problem.value)*objective_scale
    if not np.isfinite(value) or value <= 0 or not np.isfinite(optimizer_value) or optimizer_value <= 0:
        raise RuntimeError('Cannot compute collision entropy from this optimum.')
    H = -2*math.log2(value)
    recovery_gap_bits = 2*abs(math.log2(value)-math.log2(optimizer_value))
    tail_bound_bits = 2*math.log1p(tail_error/value)/math.log(2)
    psd_violation = max(psd_violation, spectral_violation)
    if max(psd_violation, norm_violation) > 100*tol or recovery_gap_bits > max(1e-7, 10*tol):
        raise RuntimeError('Collision solution failed normalized feasibility or objective recovery '
                           f'(PSD {psd_violation:.3g}, norm {norm_violation:.3g}, '
                           f'objective difference {recovery_gap_bits:.3g} bits).')
    return dict(q=2.**(-H), H_bits=H, status=problem.status,
                solve_time=time.perf_counter()-started, compilation_seconds=float(problem.compilation_time or 0.),
                solver_seconds=problem.solver_stats.solve_time, num_iters=problem.solver_stats.num_iters,
                matrix_dimension=n+1, sector_count=size, psd_violation=psd_violation,
                norm_violation=norm_violation, tail_bound_bits=tail_bound_bits, recovery_gap_bits=recovery_gap_bits)

In [ ]:
# @title 10. Baseline Collision Entropy Solver
def sanity_measured_smooth_collision_entropy(
    rho_blocks,
    epsilon=0.001,
    verbose=False,
    solver="CLARABEL",
    tol=1e-8,
):

    """
    Compute H_2^{epsilon,M,up}(X|E) using Eq. (B8).
    rho_blocks[x] = p(x) rho_{E|x}.
    """

    require_solver(solver)
    if not np.isfinite(epsilon) or not 0 <= epsilon < 1:
        raise ValueError("Require 0 <= epsilon < 1.")
    nX = len(rho_blocks)
    dE = rho_blocks[0].shape[0]
    I = np.eye(dE)

    # Variables
    B = [cp.Variable((dE, dE), hermitian=True) for _ in range(nX)]
    C = [cp.Variable((dE, dE), hermitian=True) for _ in range(nX)]
    t = cp.Variable()
    k = cp.Variable()

    # Constraints
    constraints = []
    for x in range(nX):
        constraints += [
            B[x] >> 0,
            B[x] << t * I,
            cp.bmat([
                [C[x], B[x]],
                [B[x], I],
            ]) >> 0,
        ]

    constraints += [
        sum(C) << k * I
    ]

    # Objective
    trace_term = sum(
        cp.real(cp.trace(B[x] @ rho_blocks[x]))
        for x in range(nX)
    )

    objective = cp.Maximize(
        2 * trace_term - 2 * epsilon * t - k
    )

    problem = cp.Problem(objective, constraints)

    # Solve
    problem.solve(
        solver=solver.upper(),
        verbose=verbose,
        canon_backend=cp.SCIPY_CANON_BACKEND,
        **solver_options(solver, tol),
    )


    if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        raise RuntimeError(f"Solver status: {problem.status}")
    Q_star = float(problem.value)
    if not np.isfinite(Q_star) or Q_star <= 0:
        raise RuntimeError("Cannot compute entropy from a nonpositive/nonfinite optimum.")
    H2 = -np.log2(Q_star)

    return {
        "entropy": H2,
        "Q_star": Q_star,
        "t": t.value,
        "k": k.value,
        "status": problem.status,
    }


# @title 11. Smooth Min-Entropy: Pure-State Reduction

Let $\delta$ be the **actual purified-distance radius**. Twirling the smoothed
CQ state gives $\tilde\tau_x=d^{-1}Z_xAZ_x$, where
$\operatorname{Tr}A\le1$ and $\langle v|A|v\rangle\ge1-\delta^2$.
The guessing-probability dual is a diagonal matrix $D\succeq A$, with objective
$\operatorname{Tr}D/d$. For any feasible A with nonzero overlap,
$a=Av/\sqrt{v^TAv}$ satisfies $aa^T\preceq A$, preserves the overlap and reduces
the trace. Replacing a by its componentwise magnitude only helps, since v is
nonnegative. For fixed a, minimizing the diagonal majorant gives
$\min_{D\succeq aa^T}\operatorname{Tr}D=(\sum_j a_j)^2$.

Average a over copy permutations and group by Hamming weight k. For
$m_k=\binom nk$, $u_k=\sqrt{m_kp^k(1-p)^{n-k}}$, and
$c_k=\sqrt{m_k/2^n}$, this yields
$$s^*=\min_{b\ge0}\{c^Tb:\|b\|_2\le1,\ u^Tb\ge\sqrt{1-\delta^2}\},
\qquad H_{\min}^{\delta}=-2\log_2s^*.$$
This is an n+1-variable second-order-cone problem. Its active solution has
$b\propto(u-tc)_+$, so a one-dimensional search finds the threshold t.
The implementation swaps the two basis labels when necessary so that p <= 1/2.
It uses a distance calculation rather than `1 - fidelity` to retain accuracy
for small radii such as 1e-6. It brackets a feasible threshold before bisection
so the search resolves the optimum at larger copy counts. The min-entropy functions take **radius squared** as
an argument, preserving the source notebook's calling convention.

At zero radius, the smoothed state equals the source exactly. The baseline
therefore solves the unsmoothed guessing-probability SDP directly. The conic
formulation has the unique feasible vector b=u and returns (c^T u)^2 directly.
These exact special cases avoid the degenerate fidelity constraints that can
make MOSEK terminate with UNKNOWN; no smoothing parameter is perturbed.


In [ ]:
# @title 12. Scalar, Conic, and Baseline Min-Entropy Solvers
def min_vectors(p, n):
    u, c, _ = hamming_vectors(float(min(p, 1-p)), int(n))
    return u, c


def solve_pauli_min_entropy(p, n, radius_squared):
    validate_parameters(p, n, radius_squared)
    started = time.perf_counter()
    result = dict(_solve_pauli_min_cached(float(min(p, 1-p)), int(n), float(radius_squared)))
    result['solve_time'] = time.perf_counter()-started
    return result


@lru_cache(maxsize=512)
def _solve_pauli_min_cached(p, n, radius_squared):
    validate_parameters(p, n, radius_squared)
    started = time.perf_counter()
    r2 = float(radius_squared)
    if r2 == 0:
        H = unsmoothed_min_entropy(p, n)
        return dict(H_bits=H, value=2.**(-H), status='analytic_unsmoothed',
                    solve_time=time.perf_counter()-started, fidelity_violation=0.)
    p = min(p, 1-p)
    f = math.sqrt(1-r2)
    if p in (0., 0.5):
        H = (n if p == 0 else 0)-math.log2(1-r2)
        return dict(H_bits=H, value=2.**(-H), status='analytic_endpoint',
                    solve_time=time.perf_counter()-started, fidelity_violation=0.)
    u, c = min_vectors(p, n)
    if f <= u[0]:
        b = np.zeros(n+1)
        b[0] = f/u[0]
        residual = max(0., f-float(u@b))
    else:
        target_distance_squared = 2*r2/(1+f)
        def threshold_trial(threshold):
            raw = np.maximum(u-threshold*c, 0.)
            norm = np.linalg.norm(raw)
            if norm == 0.:
                return math.inf, None
            candidate = raw/norm
            delta = u-candidate
            return float(delta@delta), candidate

        if c[0] == 0.:
            raise RuntimeError('Scalar min-entropy sector cap underflowed; reduce n.')
        hi = float(u[0])/float(c[0])
        if not math.isfinite(hi):
            raise RuntimeError('Scalar min-entropy threshold overflowed; reduce n.')

        # First find a feasible threshold at the scale of the root.
        lo = hi/2
        distance_squared, b = threshold_trial(lo)
        while lo > 0. and distance_squared > target_distance_squared:
            hi, lo = lo, lo/2
            distance_squared, b = threshold_trial(lo)

        # Keep the feasible (low-threshold) side of the bracket.
        for _ in range(110):
            mid = lo+(hi-lo)/2
            if mid == lo or mid == hi:
                break
            trial_distance, candidate = threshold_trial(mid)
            if trial_distance > target_distance_squared:
                hi = mid
            else:
                lo, distance_squared, b = mid, trial_distance, candidate
        residual = max(0., distance_squared-target_distance_squared)
    amplitude = float(c@b)
    if not np.isfinite(amplitude) or amplitude <= 0:
        raise RuntimeError('Scalar min-entropy lost objective precision; reduce n.')
    H = -2*math.log2(amplitude)
    return dict(H_bits=H, value=amplitude**2, status='scalar_threshold',
                solve_time=time.perf_counter()-started, fidelity_violation=residual)

def solve_pauli_min_socp(p, n, radius_squared, solver='CLARABEL', tol=1e-9, verbose=False):
    validate_parameters(p, n, radius_squared)
    u, c = min_vectors(p, n)
    if radius_squared == 0:
        # Cauchy-Schwarz forces b=u exactly; a tangent cone solve is unnecessary.
        return float(c@u)**2
    b = cp.Variable(n+1, nonneg=True)
    problem = cp.Problem(cp.Minimize(c@b),
                         [cp.norm(b, 2) <= 1, u@b >= math.sqrt(1-radius_squared)])
    problem.solve(solver=solver.upper(), verbose=verbose, **solver_options(solver, tol))
    if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        raise RuntimeError(problem.status)
    return float(problem.value)**2

def solve_pauli_min_baseline(p, n, radius_squared, solver='CLARABEL', tol=1e-9, verbose=False):
    validate_parameters(p, n, radius_squared, matrix=True)
    blocks = generate_pauli_cq_blocks(p, n)
    d = 2**n
    S = cp.Variable((d,d), symmetric=True)
    if radius_squared == 0:
        # Fidelity=1 and trace<=1 force the smoothed state to equal the source.
        # Use the equivalent guessing-probability SDP, with a strict interior,
        # instead of solving the degenerate zero-radius fidelity constraints.
        constraints = [S >> 0] + [S-B >> 0 for B in blocks]
    else:
        taus = [cp.Variable((d,d), symmetric=True) for _ in blocks]
        Ls = [cp.Variable((d,d)) for _ in blocks]
        constraints = [S >> 0]
        for B, tau, L in zip(blocks, taus, Ls):
            constraints += [S-tau >> 0, cp.bmat([[B,L],[L.T,tau]]) >> 0]
        constraints += [sum(cp.trace(t) for t in taus) <= 1,
                        sum(cp.trace(L) for L in Ls) >= math.sqrt(1-radius_squared)]
    problem = cp.Problem(cp.Minimize(cp.trace(S)), constraints)
    problem.solve(solver=solver.upper(), verbose=verbose, canon_backend=cp.SCIPY_CANON_BACKEND,
                  **solver_options(solver, tol))
    if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        raise RuntimeError(problem.status)
    return float(problem.value)


In [ ]:
# @title 13. Validate Reductions Against Full Optimizations
validation_rows = []
if RUN_VALIDATION:
    for p, n in itertools.product(VALIDATION_P_VALUES, VALIDATION_N_VALUES):
        if n > 2:
            raise ValueError('Keep the full baseline at n <= 2.')
        blocks = generate_pauli_cq_blocks(p, n)
        np.testing.assert_allclose(sum(np.trace(b) for b in blocks), 1., atol=1e-12)
        np.testing.assert_allclose(sum(blocks), np.diag(np.diag(sum(blocks))), atol=1e-12)
        for epsilon in VALIDATION_EPSILON_VALUES:
            reduced = solve_pauli_collision(p, epsilon, n, COLLISION_SOLVER, COLLISION_TOL, verbose=SOLVER_VERBOSE)
            baseline = sanity_measured_smooth_collision_entropy(
                blocks, epsilon, solver=COLLISION_SOLVER, tol=COLLISION_TOL, verbose=SOLVER_VERBOSE)
            np.testing.assert_allclose(reduced['q'], baseline['Q_star'],
                                       rtol=VALIDATION_RTOL, atol=VALIDATION_ATOL)
            r2 = epsilon  # Validation of the source notebook's squared-radius convention
            scalar = solve_pauli_min_entropy(p, n, r2)
            baseline_min = solve_pauli_min_baseline(p, n, r2, COLLISION_SOLVER, verbose=SOLVER_VERBOSE)
            conic_min = solve_pauli_min_socp(p, n, r2, COLLISION_SOLVER, verbose=SOLVER_VERBOSE)
            np.testing.assert_allclose(scalar['value'], baseline_min,
                                       rtol=VALIDATION_RTOL, atol=VALIDATION_ATOL)
            np.testing.assert_allclose(scalar['value'], conic_min,
                                       rtol=VALIDATION_RTOL, atol=VALIDATION_ATOL)
            validation_rows.append(dict(p=p,n=n,epsilon=epsilon,
                collision_abs_difference=abs(reduced['q']-baseline['Q_star']),
                min_abs_difference=abs(scalar['value']-baseline_min)))
    # Compare an independent conic formulation at n=3 for a nondegenerate case.
    for r2 in (0.01, 0.1, 0.9):
        np.testing.assert_allclose(solve_pauli_min_entropy(.05,3,r2)['value'],
                                   solve_pauli_min_socp(.05,3,r2,COLLISION_SOLVER,verbose=SOLVER_VERBOSE), rtol=VALIDATION_RTOL, atol=VALIDATION_ATOL)
    # Tiny purified-distance radius: check monotonicity without a near-tangent SDP.
    for n in (1,2,6):
        hs = [solve_pauli_min_entropy(.05,n,r2)['H_bits'] for r2 in (0.,1e-12,1e-6,.01)]
        assert np.all(np.diff(hs) >= -1e-11), hs
    validation_results = pd.DataFrame(validation_rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    validation_results.to_csv(OUTPUT_DIR/'validation_results.csv', index=False)
    print(f'{len(validation_rows)} baseline comparisons passed for each entropy.')
    print(validation_results[['collision_abs_difference','min_abs_difference']].max().to_string())
else:
    print('Full baseline validation skipped. Set RUN_VALIDATION=True after changing a formulation.')


### Optional collision solver diagnostic

The original MOSEK checkpoint for p = 0.05, epsilon = 0 and n = 155 records
a solver failure. To investigate one case, run the following in a new cell
after configuring the solver above. This uses the scaled formulation above.

```python
solve_pauli_collision(p=0.05, epsilon=0.0, n=155, solver=COLLISION_SOLVER,
                      tol=COLLISION_TOL, verbose=True)
```

In [ ]:
# @title 14a. Checkpoint Row Builders
def collision_case(case):
    r = solve_pauli_collision(**case, verbose=SOLVER_VERBOSE)
    return dict(q=r['q'], H_bits=r['H_bits'], bits_per_copy=r['H_bits']/case['n'],
                solver_status=r['status'], solve_seconds=r['solve_time'],
                compilation_seconds=r['compilation_seconds'], solver_seconds=r['solver_seconds'],
                num_iters=r['num_iters'], matrix_dimension=r['matrix_dimension'],
                psd_violation=r['psd_violation'], norm_violation=r['norm_violation'],
                collision_formulation=COLLISION_FORMULATION, sector_count=r['sector_count'],
                tail_bound_bits=r['tail_bound_bits'], recovery_gap_bits=r['recovery_gap_bits'])

def min_entropy_case(case):
    p, n, epsilon = case['p'], case['n'], case['epsilon']
    result = solve_pauli_min_entropy(p, n, epsilon)  # radius_squared = epsilon
    return dict(H_bits=result['H_bits'], bits_per_copy=result['H_bits']/n,
                radius=math.sqrt(epsilon), radius_squared=epsilon,
                solver_status=result['status'], solve_seconds=result['solve_time'])

In [ ]:
# @title 14b. Resume Sweeps and Load Results
if RUN_SWEEPS:
    run_sweep('collision',
        quantum_grid(COLLISION_EPSILON_VALUES, COLLISION_N_VALUES), collision_case)
    run_sweep('min_entropy',
        quantum_grid(MIN_ENTROPY_EPSILON_VALUES, MIN_ENTROPY_N_VALUES), min_entropy_case)
    reference_results.to_csv(OUTPUT_DIR / 'reference_results.csv', index=False)


def load_results(calculation, epsilon_values):
    path = checkpoint_source(calculation)
    frame = read_checkpoint(path)
    selected = frame.loc[
        frame['p'].isin(P_VALUES)
        & frame['epsilon'].isin(epsilon_values)
        & frame['solver'].eq(COLLISION_SOLVER)
        & frame['tol'].eq(COLLISION_TOL)
    ].copy()
    if selected.empty:
        raise ValueError(f'No matching {calculation} rows in {path}. '
                         'Check the settings or run a new sweep.')
    print(f'{calculation}: {len(selected)} rows from {path}')
    return selected


if REPAIR_COLLISION_CHECKPOINTS:
    previous = load_results('collision', COLLISION_EPSILON_VALUES)
    needs_repair = (previous['run_status'].ne('ok')
                    | ~previous['solver_status'].isin(('optimal', 'analytic_endpoint'))
                    | ~collision_residuals_ok(previous))
    repair_cases = previous.loc[needs_repair, ['p', 'epsilon', 'n', 'solver', 'tol']].to_dict('records')
    if repair_cases:
        if not RESUME:
            raise ValueError('Set RESUME=True to preserve the saved rows when repairing checkpoints.')
        run_sweep('collision', repair_cases, collision_case)
    else:
        print('No collision checkpoints need repair.')

collision_results = load_results('collision', COLLISION_EPSILON_VALUES)
min_entropy_results = load_results('min_entropy', MIN_ENTROPY_EPSILON_VALUES)

In [ ]:
# 15. Screen Saved Results and Plot Collision Entropy
DOT_SIZE = 2
saved_collision = successful_results(collision_results, 'Quantum collision entropy')
passes_screen = (collision_residuals_ok(saved_collision)
                 & saved_collision['solver_status'].isin(('optimal', 'analytic_endpoint')))
quantum = saved_collision.loc[passes_screen].copy()
flagged_collision = saved_collision.loc[~passes_screen].copy()
minimum = successful_results(min_entropy_results, 'Smooth min-entropy')

comparison = quantum.merge(minimum, on=['p', 'epsilon', 'n'],
                           suffixes=('_collision', '_min'), validate='one_to_one')
comparison['difference_total'] = comparison['H_bits_collision'] - comparison['H_bits_min']
comparison['difference_per_copy'] = comparison['difference_total'] / comparison['n']
comparison.to_csv(OUTPUT_DIR / 'collision_min_comparison.csv', index=False)

coverage = pd.concat({
    'Collision retained': quantum.groupby(['p', 'epsilon']).size(),
    'Collision flagged': flagged_collision.groupby(['p', 'epsilon']).size(),
    'Collision errors': collision_results.loc[collision_results['run_status'].eq('error')]
                                         .groupby(['p', 'epsilon']).size(),
    'Min-entropy successful': minimum.groupby(['p', 'epsilon']).size(),
    'Matched comparisons': comparison.groupby(['p', 'epsilon']).size(),
}, axis=1).fillna(0).astype(int).sort_index()
display(coverage)
if 'collision_formulation' in quantum:
    scaled = quantum.loc[quantum['collision_formulation'].eq(COLLISION_FORMULATION)]
    if not scaled.empty:
        print(f'{len(scaled)} scaled-solver rows retained. '
              f'Largest sector omission bound: {scaled["tail_bound_bits"].max():.3g} bits.')
if not flagged_collision.empty:
    print('Flagged collision rows are available in flagged_collision.')

for p, group in quantum.groupby('p'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for epsilon, curve in group.groupby('epsilon'):
        curve = plot_curve(curve)
        label = f'epsilon={epsilon:g}'
        axes[0].plot(curve['n'], curve['H_bits'], 'o-', label=label, markersize=DOT_SIZE)
        axes[1].plot(curve['n'], curve['bits_per_copy'], 'o-', label=label, markersize=DOT_SIZE)
    axes[0].set(title='Total collision entropy', xlabel='n (copies)', ylabel='Entropy (bits)')
    axes[1].set(title='Collision entropy per copy', xlabel='n (copies)', ylabel='Bits / copy')
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle(f'Pauli; p={p:g}')
    fig.tight_layout()
    save_figure(fig, f'collision_entropy_p{p:g}')

In [ ]:
# @title 16. Collision Entropy vs Min-Entropy: Comparison Plots
for p, group in comparison.groupby('p'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for idx, (epsilon, curve) in enumerate(group.groupby('epsilon')):
        curve = plot_curve(curve)
        color = plt.cm.tab10.colors[idx % 10]
        label_coll = f'Collision: epsilon={epsilon:g}'
        label_min = f'Min: epsilon={epsilon:g}'
        axes[0].plot(curve['n'], curve['H_bits_collision'], 'o-', color=color, label=label_coll, markersize=DOT_SIZE)
        axes[1].plot(curve['n'], curve['bits_per_copy_collision'], 'o-', color=color, label=label_coll, markersize=DOT_SIZE)
        axes[0].plot(curve['n'], curve['H_bits_min'], 'v--', color=color, label=label_min, markersize=DOT_SIZE)
        axes[1].plot(curve['n'], curve['bits_per_copy_min'], 'v--', color=color, label=label_min, markersize=DOT_SIZE)
    axes[0].set(title='Total entropy', xlabel='n (copies)', ylabel='Entropy (bits)')
    axes[1].set(title='Entropy per copy', xlabel='n (copies)', ylabel='Bits / copy')
    for index, ax in enumerate(axes):
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, ncol=2, loc='upper left' if index == 0 else 'upper right')
    fig.suptitle(f'Pauli; p={p:g}.')
    fig.tight_layout()
    save_figure(fig, f'entropy_comparison_p{p:g}')

In [ ]:
# @title 17. Collision Entropy vs Min-Entropy: Difference Plots

for p, group in comparison.groupby('p'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for idx, (epsilon, curve) in enumerate(group.groupby('epsilon')):
        curve = plot_curve(curve)
        color = plt.cm.tab10.colors[idx % 10]
        label = f'epsilon={epsilon:g}'
        axes[0].plot(curve['n'], curve['difference_total'], 'o-', color=color, label=label, markersize=DOT_SIZE)
        axes[1].plot(curve['n'], curve['difference_per_copy'], 'o-', color=color, label=label, markersize=DOT_SIZE)
    axes[0].set(
        title=r'$H_2^{\varepsilon, \mathbb{M}} - H_{\min}^{\sqrt{\varepsilon}, \mathbb{P}}$ (bits)',
        xlabel='n (copies)', ylabel=r'$\Delta H$ (bits)')
    axes[1].set(
        title='Difference in Entropy per Copy',
        xlabel='n (copies)', ylabel=r'$\Delta (\mathrm{bits / copy})$')
    for index, ax in enumerate(axes):
        ax.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
        ax.grid(alpha=0.3)
        if index == 0:
            ax.legend(fontsize=8, loc='upper left')
        else:
            ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1))
    fig.suptitle(f'Entropy Difference (Collision - Min-Entropy); Pauli p={p:g}')
    fig.tight_layout()
    save_figure(fig, f'entropy_difference_p{p:g}')

In [ ]:
# 17b. Entropy Difference for the First Five Copy Counts
X_LIMITS = (0, 5)

for p, group in comparison.groupby('p'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for idx, (epsilon, curve) in enumerate(group.groupby('epsilon')):
        curve = plot_curve(curve)
        color = plt.cm.tab10.colors[idx % 10]
        label = f'epsilon={epsilon:g}'
        axes[0].plot(curve['n'], curve['difference_total'], 'o-', color=color, label=label, markersize=DOT_SIZE)
        axes[1].plot(curve['n'], curve['difference_per_copy'], 'o-', color=color, label=label, markersize=DOT_SIZE)
    axes[0].set(
        title=r'$H_2^{\varepsilon, \mathbb{M}} - H_{\min}^{\sqrt{\varepsilon}, \mathbb{P}}$ (bits)',
        xlabel='n (copies)', ylabel=r'$\Delta H$ (bits)')
    axes[1].set(
        title='Difference in Entropy per Copy',
        xlabel='n (copies)', ylabel=r'$\Delta (\mathrm{bits / copy})$')
    for index, ax in enumerate(axes):
        ax.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
        ax.grid(alpha=0.3)
        ax.set_ylim(-0.5,0.5)
        ax.set_xlim(X_LIMITS)
        if index == 0:
            ax.legend(fontsize=8, loc='upper left')
        else:
            ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1))
    fig.suptitle(f'Entropy Difference (Collision - Min-Entropy); Pauli p={p:g}')
    fig.tight_layout()
    save_figure(fig, f'entropy_difference_zoom_p{p:g}')

## Reading and extending the results

- `collision_results` and `min_entropy_results` contain the selected checkpoint
  rows, including recorded failures. The plot uses `quantum`, the collision rows
  that succeeded and whose two stored feasibility residuals are finite and no
  greater than 100 times the row's solver tolerance. This is the warning threshold
  used in the collision solver. Inspect excluded successful rows in
  `flagged_collision`. Passing the screen provides a feasibility check.
- `comparison` pairs the screened collision rows with successful min-entropy rows
  at the same p, epsilon and n. Lines break across missing or screened-out copy
  counts. The full difference plot and the first-five-copy view have separate files.
- The saved-results view uses all available copy counts for the selected series.
  The copy-count settings control new sweeps. With `RUN_SWEEPS = True` and
  `RESUME = True`, successful cases with the same model, parameters, solver and
  tolerance are reused, and failed cases in the requested grid are retried.
  Collision sweeps also recompute requested rows from the original solver and
  retry inaccurate solutions or rows above the feasibility threshold.
- Set `REPAIR_COLLISION_CHECKPOINTS = True` to retry just the failed and
  flagged cases across the selected saved collision series. The repaired rows
  are tagged `normalized-gauge-v1`, with residuals in scaled units and separate
  columns for the sector omission bound and objective recovery difference.
  Earlier checkpoint residuals are stored in the original solver's units.

Both tables use `H_bits` for total entropy and `bits_per_copy` for the rate.
Min-entropy uses the purified-distance radius delta = sqrt(epsilon), so its
`radius_squared` equals epsilon. The merged table distinguishes the quantities
with `_collision` and `_min` suffixes. All logarithms are base 2.

The collision solve uses a matrix of size n+1. Larger copy counts require
checks of conditioning and feasibility residuals. Full baseline validation
is restricted to n <= 2 and is available via `RUN_VALIDATION = True`.